# Cálculo del Jacobiano simbólico para el estudio de las singularidades del IRB140

In [ ]:
# Para correr en Colab
%pip install roboticstoolbox-python
%pip install numpy==1.26.4 --force-reinstall

# Nota importante: da error porque un montón de bibliotecas de Colab dependen de numpy 2, 
# pero como no se utilizan no importa. 
#
# Cuando termina de ejecutarse la celda, pide reiniciar el kernel (aparece un botón bien evidente)
# Luego de reiniciar correr las celdas siguientes

In [2]:
import sympy as sp
import numpy as np
import roboticstoolbox as rtb

## Clase RobotSimbolico

Extiende las capacidades de roboticstoolbox para el cálculo del jacobiano simbólico
- Permite definir el punto de cálculo de las velocidades
- Permite expresar en ternas arbitrarias tanto la velocidad de traslación y rotación


In [82]:
class RobotSimbolico(rtb.DHRobot):
    def __init__(self, links, **kwargs):
        super().__init__(links, **kwargs)

    def fkine_symbolic(self, symbolic_lengths=True):
        """
        Devuelve:
            A_list: lista de matrices homogéneas A_i simbólicas
            T_list: lista de transformaciones acumuladas
            q: vector simbólico de variables articulares
        """


        # Variables articulares
        q = sp.symbols(f'q1:{self.n+1}', real=True)

        A_list = []
        T_list = []

        T = sp.eye(4)

        for i, link in enumerate(self.links):

            # Parámetros DH desde rtb
            a_i = link.a
            d_i = link.d
            alpha_i = link.alpha
            theta_i = link.theta
            sigma = link.sigma  # 0: revoluta, 1: prismática

            # --- Manejo simbólico de longitudes ---
            if symbolic_lengths:
                if a_i != 0:
                    a_i = sp.symbols(f'a{i+1}', real=True)
                else:
                    a_i = 0

                if d_i != 0:
                    d_i = sp.symbols(f'd{i+1}', real=True)
                else:
                    d_i = 0
            else:
                a_i = float(a_i)
                d_i = float(d_i)

            # --- Variables articulares ---
            if sigma == 0:  # revoluta
                theta = q[i]
                d = d_i
            else:  # prismática
                theta = theta_i
                d = q[i]

            # --- Matriz DH ---
            A_i = sp.Matrix([
                [sp.cos(theta), -sp.sin(theta)*sp.cos(alpha_i),  sp.sin(theta)*sp.sin(alpha_i), a_i*sp.cos(theta)],
                [sp.sin(theta),  sp.cos(theta)*sp.cos(alpha_i), -sp.cos(theta)*sp.sin(alpha_i), a_i*sp.sin(theta)],
                [0,              sp.sin(alpha_i),                sp.cos(alpha_i),               d],
                [0,              0,                              0,                             1]
            ])

            # Limpieza tipo tu MATLAB (ceros numéricos chicos)
            A_i = A_i.applyfunc(kill_small_terms)
            A_list.append(A_i)

            # Acumulada
            T = (T @ A_i).applyfunc(sp.simplify)
            T_list.append(T)

        return A_list, T_list, q
    
    def calc_Aij(self,A, i, j, simplify_result=False):
        """
        Calcula la transformación entre i y j usando lista de matrices A_i.

        Parámetros:
            A : list of sympy.Matrix (A[0] = A1 en MATLAB)
            i, j : índices (base 0)
            simplify_result : bool (opcional, evitar por defecto)

        Retorna:
            Aij : sympy.Matrix
        """

        # convertir a índices Python
        i -= 1
        j -= 1

        # Determinar sentido
        if i < j:
            ss = -1
            k_idx = list(range(i, j+1))
        elif i > j:
            ss = 1
            k_idx = list(range(j, i+1))
        else:
            return sp.eye(4)

        # Producto
        Aij = sp.eye(4)

        for k in range(1, len(k_idx)):
            Aij = Aij @ A[k_idx[k]]

        # Inversión si corresponde
        if ss == -1:
            Aij = inv_SE3(Aij)

        # Simplificación opcional (controlada)
        if simplify_result:
            Aij = Aij.applyfunc(sp.simplify)

        return Aij


    def calc_jacobian(self,A, oJv, pJv, pJw, clean=False):
        """
        Jacobiano simbólico usando directamente un robot de rtb

        Parámetros:
            A : lista de matrices homogéneas (A[0] = A1)
            robot : objeto de roboticstoolbox
            oJv, pJv, pJw : índices de frames (base 0)
        """

        J = sp.zeros(6, self.n)

        for iq in range(self.n):

            link = self.links[iq]
            is_revolute = (link.sigma == 0)

            Aojv = self.calc_Aij(A, oJv, iq)
            Apjv = self.calc_Aij(A, pJv, iq)
            Apjw = self.calc_Aij(A, pJw, iq)

            # --- bloque lineal ---
            p = Aojv[:3, 3]

            for ic in range(3):
                if is_revolute:
                    r = Apjv[:3, ic]
                    v_aux = p.cross(r)
                else:
                    v_aux = Apjw[:3, ic]

                J[ic, iq] = v_aux[2]

            # --- bloque angular ---
            for ic in range(3):
                if is_revolute:
                    r = Apjw[:3, ic]
                    J[ic+3, iq] = r[2]
                else:
                    J[ic+3, iq] = 0

        if clean:
            J = J.applyfunc(lambda e: clean_expr(e, 1e-12))
        return J

def clean_expr(expr, tol=1e-10):
    # Primero intento simplificar trigonométrico
    expr = sp.trigsimp(expr)

    # Si queda numérico chico → cero
    if expr.is_number:
        if abs(complex(expr)) < tol:
            return sp.Integer(0)

    return expr

def clean_small_numeric(expr, tol=1e-10):
    if expr.is_number:
        if abs(complex(expr)) < tol:
            return sp.Integer(0)
    return expr

import sympy as sp

def kill_small_terms(expr, tol=1e-12):
    # separa coeficiente y parte simbólica
    coeff, rest = expr.as_coeff_Mul()

    try:
        if abs(complex(coeff)) < tol:
            return sp.Integer(0)
    except:
        pass

    return expr

def substitute_lengths(expr, robot):
    subs = {}
    for i, link in enumerate(robot.links):
        if link.a != 0:
            subs[sp.symbols(f'a{i}')] = link.a
        if link.d != 0:
            subs[sp.symbols(f'd{i}')] = link.d
    return expr.subs(subs)


def inv_SE3(T):
    R = T[:3, :3]
    p = T[:3, 3]

    R_T = R.T
    p_new = -R_T @ p

    T_inv = sp.eye(4)
    T_inv[:3, :3] = R_T
    T_inv[:3, 3] = p_new

    return T_inv




## Definición del IRB140 y análisis de sus singularidades

In [83]:
class IRB140(RobotSimbolico):

    def __init__(self):
        links = [
            rtb.RevoluteDH(alpha=-np.pi/2, a=0.07, d=0.352),
            rtb.RevoluteDH(a=0.36, offset=-np.pi/2),
            rtb.RevoluteDH(alpha=np.pi/2, offset=np.pi),
            rtb.RevoluteDH(d=0.38, alpha=-np.pi/2),
            rtb.RevoluteDH(alpha=np.pi/2),
            rtb.RevoluteDH(d=0)
        ]

        super().__init__(links, name="IRB140")
        
robot = IRB140()
print(robot)


DHRobot: IRB140, 6 joints (RRRRRR), dynamics, standard DH parameters
┌────────────┬───────┬──────┬────────┐
│     θⱼ     │  dⱼ   │  aⱼ  │   ⍺ⱼ   │
├────────────┼───────┼──────┼────────┤
│  q1        │ 0.352 │ 0.07 │ -90.0° │
│  q2 - 90°  │     0 │ 0.36 │   0.0° │
│  q3 + 180° │     0 │    0 │  90.0° │
│  q4        │  0.38 │    0 │ -90.0° │
│  q5        │     0 │    0 │  90.0° │
│  q6        │     0 │    0 │   0.0° │
└────────────┴───────┴──────┴────────┘

┌──┬──┐
└──┴──┘



In [84]:
# Calculo la cinemática simbólica
A_list, T_list, q = robot.fkine_symbolic(symbolic_lengths=True)


In [85]:
A_ij = robot.calc_Aij(A_list,0,1)
sp.pprint(A_ij)
sp.pprint(sp.simplify(A_ij@A_list[0]))

A_ij = robot.calc_Aij(A_list,1,0)
sp.pprint(A_ij)



⎡                                         2             2    ⎤
⎢  cos(q₁)       sin(q₁)     0    - a₁⋅sin (q₁) - a₁⋅cos (q₁)⎥
⎢                                                            ⎥
⎢     0             0       -1.0            1.0⋅d₁           ⎥
⎢                                                            ⎥
⎢-1.0⋅sin(q₁)  1.0⋅cos(q₁)   0                 0             ⎥
⎢                                                            ⎥
⎣     0             0        0                 1             ⎦
⎡1   0    0   0⎤
⎢              ⎥
⎢0  1.0   0   0⎥
⎢              ⎥
⎢0   0   1.0  0⎥
⎢              ⎥
⎣0   0    0   1⎦
⎡cos(q₁)   0    -1.0⋅sin(q₁)  a₁⋅cos(q₁)⎤
⎢                                       ⎥
⎢sin(q₁)   0    1.0⋅cos(q₁)   a₁⋅sin(q₁)⎥
⎢                                       ⎥
⎢   0     -1.0       0            d₁    ⎥
⎢                                       ⎥
⎣   0      0         0            1     ⎦


In [91]:
J= robot.calc_jacobian(A_list,4,3,4,6)

In [92]:
J11 = J[0:3,0:3]
J11=sp.simplify(J11)
sp.pprint(J11)
print("\nDeterminante ",sp.det(J11))

⎡                      0                        1.0⋅a₂⋅sin(q₃) + 1.0⋅d₄  1.0⋅d₄⎤
⎢                                                                              ⎥
⎢1.0⋅a₁ + 1.0⋅a₂⋅cos(q₂) + 1.0⋅d₄⋅sin(q₂ + q₃)             0               0   ⎥
⎢                                                                              ⎥
⎣                      0                            -1.0⋅a₂⋅cos(q₃)        0   ⎦

Determinante  -1.0*a1*a2*d4*cos(q3) - 1.0*a2**2*d4*cos(q2)*cos(q3) - 1.0*a2*d4**2*sin(q2 + q3)*cos(q3)


In [88]:
J22 = J[3:,3:]
J22=sp.simplify(J22)
sp.pprint(J22)

print("\nDeterminante ",sp.det(J22))

⎡ 0    0  1.0⋅sin(q₅) ⎤
⎢                     ⎥
⎢-1.0  0  -1.0⋅cos(q₅)⎥
⎢                     ⎥
⎣ 0    1       0      ⎦

Determinante  -1.0*sin(q5)
